# Azure Observable RAG — Demo Notebook

Walks one query end-to-end through the LangGraph DAG and renders every node's structured trace.

**Prereqs**
1. `bash infra/deploy.sh <rg> <location>` has been run (or `.env` is otherwise populated).
2. Sample documents have been ingested: `python -m src.cli ingest`.


In [ ]:
import os, sys, json
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

import pandas as pd
from src.agent import run
from src.search import hybrid_search

## 1 · The Query

In [ ]:
QUERY = 'How do I factory-reset Device A?'
QUERY

## 2 · Run the LangGraph and capture the FinalRagTrace

In [ ]:
trace = run(QUERY)
print('total_latency_ms:', trace['total_latency_ms'])
print('intent:', trace['query_plan']['intent'])

## 3 · Query Plan

In [ ]:
qp = trace['query_plan']
pd.DataFrame([{
    'original':     qp['original_query'],
    'rewritten':    qp['rewritten_query'],
    'intent':       qp['intent'],
    'filters':      qp['filters'],
    'search_mode':  qp['search_mode'],
    'top_k':        qp['top_k'],
    'rationale':    qp.get('notes'),
}])

## 4 · Retrieved top-K Chunks

In [ ]:
ret = trace['retrieval']
print(f"latency: {ret['latency_ms']} ms  ·  results: {len(ret['results'])}")
df = pd.DataFrame(ret['results'])[[
    'rank', 'score', 'reranker_score', 'file_name', 'page_number',
    'heading_path', 'semantic_caption', 'chunk_id'
]]
df['chunk_id'] = df['chunk_id'].str[:12] + '…'
df

## 5 · Selected Evidence

In [ ]:
ev = trace['evidence_selection']
print('strategy :', ev['selection_strategy'])
print('rationale:', ev['rationale'])
print('candidates:', ev['candidate_count'], '→ selected:', len(ev['selected_chunk_ids']))
ev['selected_chunk_ids']

## 6 · Final Generated Answer

In [ ]:
from IPython.display import Markdown
g = trace['generation']
print(f"model: {g['model']}  ·  prompt~tok: {g['prompt_token_estimate']}  ·  completion~tok: {g['completion_token_estimate']}  ·  latency: {g['latency_ms']} ms")
Markdown(g['answer'])

## 7 · Citations

In [ ]:
citations_df = pd.DataFrame(g['citations'])
citations_df['chunk_id'] = citations_df['chunk_id'].str[:12] + '…' if not citations_df.empty else citations_df.get('chunk_id')
citations_df

## 8 · Optional — retrieval-only inspection (audits the index without an LLM)

In [ ]:
raw = hybrid_search(QUERY, top_k=8, search_mode='hybrid_semantic')
raw_df = pd.DataFrame([r.to_dict() for r in raw])[[
    'rank', 'score', 'reranker_score', 'file_name', 'page_number', 'heading_path', 'content_preview'
]]
raw_df

## 9 · Full FinalRagTrace JSON

This is the same payload that gets appended to `traces.jsonl`, served to the Chainlit UI, and consumed by `eval.ipynb`.

In [ ]:
print(json.dumps(trace, indent=2, default=str)[:4000])